In [34]:
import torch
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score
from core import *
from utils import *
from lark import Tree, Token
from pm4py import save_vis_petri_net
import random
import pandas as pd
import sys
#print(f"Versione Python: {sys.version}")

#print(torch.cuda.is_available())
#print(torch.__version__)

# SETTINGS
NARY = 1
PROBABILITIES = 0.2,0.2,0.2,0.4
FILE_PATH_PNG = "petri_net_output.png"
TRACE_ENC_REG = "data/regions_full.csv"
TRACE_ENC_TAS = "data/tasks_full.csv"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
learning_rate = 3e-4

cuda


In [35]:
def get_batch_model(split, train_data, val_data, batch_size, block_size, device):
    data = train_data if split == 'train' else val_data

    ix = torch.randint(len(data) - block_size, (batch_size,)) # Prendo batch_size indici casuali

    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    x,y = x.to(device), y.to(device)
    return x, y

'''def get_batch_model_time(split, train_data, train_times, val_data, val_times, batch_size, block_size, device):
    data = train_data if split == 'train' else val_data
    time = train_times if split == 'train' else val_times

    ix = torch.randint(len(data) - block_size, (batch_size,))

    x_data = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([time[i:i+block_size] for i in ix])

    x_times = y.clone()
    x_times[:, -1] = 0.0

    x_data, x_times, y = x_data.to(device), x_times.to(device), y.to(device)
    return x_data, x_times, y'''

# VERSIONE 2 - IPOTIZZO SIA MEGLIO EH
def get_batch_model_time(split, train_data, train_times, val_data, val_times, batch_size, block_size, device):
    data = train_data if split == 'train' else val_data
    time = train_times if split == 'train' else val_times

    ix = torch.randint(1, len(data) - block_size, (batch_size,))

    x_data = torch.stack([data[i:i+block_size] for i in ix])
    x_times = torch.stack([time[i-1:i+block_size-1] for i in ix])
    y = torch.stack([time[i:i+block_size] for i in ix])

    x_data, x_times, y = x_data.to(device), x_times.to(device), y.to(device)
    return x_data, x_times, y

@torch.no_grad()
def estimate_loss(model, eval_iters, train_data, val_data, batch_size, block_size, device):
    out = {}
    model.eval() # Metto il modello in modalità eval (non train)

    # Calcolo la loss su eval_iters batch
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch_model(split, train_data, val_data, batch_size, block_size, device)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()

    model.train()
    return out

@torch.no_grad()
def estimate_loss_times(model, eval_iters, train_data, train_times, val_data, val_times, batch_size, block_size, device):
    out = {}
    model.eval() # Metto il modello in modalità eval (non train)

    # Calcolo la loss su eval_iters batch
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X_data, X_times, Y = get_batch_model_time(split, train_data, train_times, val_data, val_times, batch_size, block_size, device)
            logits, loss = model(X_data, X_times, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()

    model.train()
    return out

In [36]:
iterations = 15  # Quante iterazioni diverse (numero regioni prima di minimizzare in teoria)
current_string = SEED_STRING
for _ in range(iterations):
    current_string = replace_random_underscore(current_string, PROBABILITIES)

process = replace_underscores(current_string)
tree = PARSER.parse(process)

# ALBERI GIOCATTOLO

tree = Tree('xor', [Tree('task', [Token('NAME', 'T1')]),Tree('sequential', [Tree('parallel', [Tree('xor', [Tree('task', [Token('NAME', 'T2')]),Tree('xor', [Tree('task', [Token('NAME', 'T3')]),Tree('task', [Token('NAME', 'T4')])])]),Tree('parallel', [Tree('task', [Token('NAME', 'T5')]),Tree('task', [Token('NAME', 'T6')])])]),Tree('sequential', [Tree('task', [Token('NAME', 'T7')]),Tree('task', [Token('NAME', 'T8')])])])])

#tree = Tree('xor', [Tree('loop', [Tree('sequential', [Tree('loop', [Tree('task', [Token('NAME', 'T1')])]), Tree('xor', [Tree('task', [Token('NAME', 'T2')]), Tree('task', [Token('NAME', 'T3')])])])]), Tree('parallel', [Tree('task', [Token('NAME', 'T4')]), Tree('loop', [Tree('task', [Token('NAME', 'T5')])])])])

if NARY: # Se true allora faccio l'albero ennario (semplifico)
    tree = createNAryTree(tree)

#tree = Tree('loop', [Tree('parallel', [Tree('sequential', [Tree('loop', [Tree('parallel', [Tree('task', [Token('NAME', 'T1')]), Tree('task', [Token('NAME', 'T2')])])]), Tree('loop', [Tree('parallel', [Tree('loop', [Tree('task', [Token('NAME', 'T3')])]), Tree('task', [Token('NAME', 'T4')])])])]), Tree('loop', [Tree('sequential', [Tree('task', [Token('NAME', 'T5')]), Tree('loop', [Tree('task', [Token('NAME', 'T6')])])])])])])

print(tree)

Tree('xor', [Tree('task', [Token('NAME', 'T1')]), Tree('sequential', [Tree('parallel', [Tree('xor', [Tree('task', [Token('NAME', 'T2')]), Tree('task', [Token('NAME', 'T3')]), Tree('task', [Token('NAME', 'T4')])]), Tree('task', [Token('NAME', 'T5')]), Tree('task', [Token('NAME', 'T6')])]), Tree('task', [Token('NAME', 'T7')]), Tree('task', [Token('NAME', 'T8')])])])


In [37]:
net = PetriNetP(tree)

save_vis_petri_net(
    net.net,
    net.initial_marking,
    net.final_marking,
    FILE_PATH_PNG,
    format="png"
)

# Oggetto Generator
generator = Generator(5000, net)

In [38]:
# Creazione matrice identità delle regioni
df_region_identity = pd.DataFrame.from_dict(net.node_identity, orient='index').sort_index()
df_region_identity.columns = ['X', '+', '->', '<>']
print(df_region_identity)

# Creazione matrice regioni-figli per le regioni
df_region_children = pd.Series(net.node_children).explode()
df_region_children = pd.crosstab(df_region_children.index, df_region_children)
df_region_children = df_region_children.reindex(index=net.regions, columns=net.regions + net.tasks, fill_value=0)
df_region_children = df_region_children.astype(int)
df_region_children.index.name = None
df_region_children.columns.name = None
print(df_region_children)

# Codifica delle tracce generate
traceEncoded_regions, traceEncoded_tasks = getEncoding(generator.generatedTraces, net.regions, net.tasks, net.open_clauses, net.end_clauses)

# Trovo numero regioni e numero task effettivo (torneranno utili in futuro)
num_regions = len([i for i in traceEncoded_regions.index if str(i).startswith('R')])
num_tasks = len([i for i in traceEncoded_tasks.index if str(i).startswith('T')])

# Penso evitabili d'ora in poi
#traceEncoded_regions.to_csv(TRACE_ENC_REG, index=True)
#traceEncoded_tasks.to_csv(TRACE_ENC_TAS, index=True)

# Creo il dataframe unico (regioni + tasks)
df_traces = pd.concat([traceEncoded_regions, traceEncoded_tasks], axis=0)
print(df_traces)

df_traces = df_traces.T
df_tracescopy = df_traces.copy()

# Operazioni per creazione funzione di codifica e decodifica da colonna ad id e viceversa
unique_columns = df_traces.drop_duplicates()
unique_tuple = [tuple(x) for x in unique_columns.values]

bit_to_id = {v: i for i, v in enumerate(unique_tuple)}
id_to_bit = {i: v for i, v in enumerate(unique_tuple)}

vocab_size = len(unique_columns)

encode = lambda a: [bit_to_id[tuple(x)] for x in a]
decode = lambda b: [id_to_bit[x] for x in b]

time_intra_token = {
    (c1, c2): random.randint(1, 20) if c1 != tuple([[0]] * (num_tasks+num_regions)) else 0
    for c1 in unique_tuple
    for c2 in unique_tuple
}

times = [0]
for i in range(1, len(df_traces)):
    prev_tuple = tuple(df_traces.iloc[i-1].astype(int))
    curr_tuple = tuple(df_traces.iloc[i].astype(int))

    times.append(time_intra_token[(prev_tuple, curr_tuple)])

# Creo il tensor con le colonne codificate in interi
data = torch.tensor(encode(df_traces.values), dtype=torch.long)
data_times = torch.tensor(times, dtype=torch.long)

    X  +  ->  <>
R0  1  0   0   0
R1  0  0   1   0
R2  0  1   0   0
R3  1  0   0   0
    R0  R1  R2  R3  T1  T2  T3  T4  T5  T6  T7  T8
R0   0   1   0   0   1   0   0   0   0   0   0   0
R1   0   0   1   0   0   0   0   0   0   0   1   1
R2   0   0   0   1   0   0   0   0   1   1   0   0
R3   0   0   0   0   0   1   1   1   0   0   0   0
    0      1      2      3      4      5      6      7      8      9      ...  \
R0      1      1      1      1      1      1      1      1      1      0  ...   
R1      1      1      1      1      1      1      1      1      1      0  ...   
R2      1      1      1      1      1      0      0      0      0      0  ...   
R3      1      0      0      0      0      0      0      0      0      0  ...   
T1      0      0      0      0      0      0      0      0      0      0  ...   
T2      0      0      0      0      0      0      0      0      0      0  ...   
T3      0      0      0      0      0      0      0      0      0      0  ...   
T4      1   

In [39]:
# Valori per il modello
block_size = 128
n_embd = 128
dropoutModel = 0.25
n_headModel = 4
n_layerModel = 2
batch_size = 16
eval_iters = 200
eval_interval = 150
max_iters = 1000
learning_rate_model = 0.0008652270052434077

dropoutTime = 0.20
n_headTime = 4
n_layerTime = 3
learning_rate_timeM = 3e-4

# Creazione modello BPMNTransformer
model = BPMNTransformer(vocab_size, block_size, n_embd, dropoutModel, n_headModel, n_layerModel)
m = model.to(device)

# Creazione modello TimeTransformer
time_model = TimeTransformer(vocab_size, block_size, n_embd, dropoutTime, n_headTime, n_layerTime)
t = time_model.to(device)

print(sum(p.numel() for p in m.parameters()) / 1e6, 'M parameters') # Calcolo parametri modelli

optimizerModel = torch.optim.AdamW(model.parameters(), lr=learning_rate_model) # Ottimizzatore pyTorch

optimizerTime = torch.optim.AdamW(time_model.parameters(), lr=learning_rate_timeM) # Ottimizzatore pyTorch

n = int(0.8 * len(df_traces)) # 80% train, 20% validation

train_data = data[:n]
val_data = data[n:]
train_times = data_times[:n]
val_times = data_times[n:]

0.415243 M parameters


In [40]:
# Funzione presa dal video di karpathy (alleno il modello Model)
max_iters=500
for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1: # Ogni tot stampo la loss corrente
        losses = estimate_loss(model, eval_iters, train_data, val_data, batch_size, block_size, device)
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch_model('train', train_data, val_data, batch_size, block_size, device) # Pesco tracce

    logits, loss = model(xb, yb) # Esegue il forward e predice
    optimizerModel.zero_grad(set_to_none=True) # Reset gradienti (puliamo i calcoli del giro precedente)

    loss.backward() #Errore per neurone
    optimizerModel.step() # Aggiorna automaticamente i pesi per sbagliare di meno al giro dopo

step 0: train loss 2.4178, val loss 2.4169
step 150: train loss 0.3910, val loss 0.3961
step 300: train loss 0.3526, val loss 0.3550
step 450: train loss 0.3509, val loss 0.3554
step 499: train loss 0.3487, val loss 0.3531


In [50]:
# Funzione presa dal video di karpathy (alleno il modello del Tempo - adattata)
max_iters = 2000
for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1: # Ogni tot stampo la loss corrente
        losses = estimate_loss_times(time_model, eval_iters, train_data, train_times, val_data, val_times, batch_size, block_size, device)
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xd, xt, y = get_batch_model_time('train', train_data, train_times, val_data, val_times, batch_size, block_size, device) # Pesco tracce

    logits, loss = time_model(xd, xt, y) # Esegue il forward e predice
    optimizerTime.zero_grad(set_to_none=True) # Reset gradienti (puliamo i calcoli del giro precedente)

    loss.backward() #Errore per neurone
    optimizerTime.step() # Aggiorna automaticamente i pesi per sbagliare di meno al giro dopo

step 0: train loss 0.5830, val loss 0.6278
step 150: train loss 0.3450, val loss 0.3508
step 300: train loss 0.2332, val loss 0.2566
step 450: train loss 0.5188, val loss 0.5881
step 600: train loss 0.2890, val loss 0.3034
step 750: train loss 0.2656, val loss 0.2740
step 900: train loss 0.2205, val loss 0.2295
step 1050: train loss 0.2067, val loss 0.2142
step 1200: train loss 0.2334, val loss 0.2404
step 1350: train loss 0.1996, val loss 0.2060
step 1500: train loss 0.2203, val loss 0.2229
step 1650: train loss 0.2263, val loss 0.2326
step 1800: train loss 0.2015, val loss 0.2046
step 1950: train loss 0.1949, val loss 0.2016
step 1999: train loss 0.1964, val loss 0.2004


In [61]:
# DA RIVEDERE LA GENERAZIONE DELLE COSE QUANDO SI AGGIUNGE IL TIMETRANSFORMER
context = torch.tensor(encode([[0]*(num_regions+num_tasks)]), dtype=torch.long, device=device).unsqueeze(0)
context_time = torch.tensor([1,1], dtype=torch.float32, device=device).unsqueeze(0)
fake_time = torch.tensor([0.0], dtype=torch.float32, device=device).unsqueeze(0)

max_new_tokens = 10
generated_indices = []
generated_times = [1]
for step in range(max_new_tokens):
    # CON MASCHERA 0 FINALE
    '''next_id = m.predict_next_task(idx=context, block_size=block_size)
    context = torch.cat((context, next_id), dim=1)
    context_time_fake = torch.cat((context_time, fake_time), dim=1)
    next_time = t.predict_next_time(idx_tasks=context, idx_times=context_time_fake, block_size=block_size)
    next_time = torch.round(next_time)
    context_time = torch.cat((context_time, next_time), dim=1)
    generated_indices.append(next_id.item())
    generated_times.append(next_time.item())'''

    # CON SLIDING WINDOW
    next_id = m.predict_next_task(idx=context, block_size=block_size)
    context = torch.cat((context, next_id), dim=1)
    next_time = t.predict_next_time(idx_tasks=context, idx_times=context_time, block_size=block_size)
    next_time = torch.round(next_time)
    context_time = torch.cat((context_time, next_time), dim=1)
    generated_indices.append(next_id.item())
    generated_times.append(next_time.item())

decoded_output = decode(generated_indices)

for i in range(len(decoded_output)):
    step = decoded_output[i]
    bit_list = [int(b) for b in step]

    time = generated_times[i+1]

    if i>0:
        real_time = time_intra_token[tuple(decoded_output[i-1]),tuple(decoded_output[i])]
    else:
        real_time = time_intra_token[tuple([0]*(num_regions+num_tasks)),tuple(decoded_output[i])]
    print(f"{bit_list} - {time} - {real_time}")


print("\n--- TRACCE GENERATE ---")
traces_generated = []
current_trace_generated = []
for i, step in enumerate(decoded_output):
    bit_list = [int(b) for b in step]
    current_trace_generated.append(bit_list)
    if bit_list == [0]*(num_regions+num_tasks):
        traces_generated.append(current_trace_generated)
        current_trace_generated = []
    #print(f"Step {i:02d}: {bit_list}")

traces_generated.remove(traces_generated[0]) #Rimuovo la prima che è sempre [], generata ed inserita dall'algoritmo sopra
for i,trace in enumerate(traces_generated):
    print(f"{i}: {trace}")


[1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0] - 2.0 - 0
[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0] - 16.0 - 16
[1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0] - 4.0 - 4
[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0] - 9.0 - 9
[1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0] - 17.0 - 18
[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] - 14.0 - 14
[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0] - 10.0 - 10
[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] - 13.0 - 13
[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1] - 1.0 - 1
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] - 12.0 - 12

--- TRACCE GENERATE ---


In [12]:
from pm4py.algo.conformance.alignments.petri_net import algorithm as alignments
from pm4py.objects.log.obj import Trace, Event, EventLog

# Facciamo il decoding delle tracce generate
#print(traces_generated)
traces_decoded = getDecoding(traces_generated, net.regions, net.tasks)
for i,trace in enumerate(traces_decoded):
    print(f"{i}: {trace}")

# Creiamo il l'EventLog di ogni traccia per poi poterla allineare
eventlog_traces = EventLog()
for trace in traces_decoded:
    t = Trace()
    for activity in trace:
        t.append(Event({'concept:name': activity}))
    eventlog_traces.append(t)


check = ["P", "X", "L"]
start = tuple(["start_" + c for c in check])
end = tuple(["end_" + c for c in check if c!="L"]) # si escludono gli end loop
loop = tuple(["back_L"]) + tuple(["end_L"])

silent_transition_prefixes = start + end + loop

model_cost_function = dict()
sync_cost_function = dict()

# METTIAMO 10000 come di default???
for t in net.net.transitions: # Prendo tutte le transizioni
    if t.label is not None and t.label.startswith(silent_transition_prefixes): # Se è una transizione silente,
        model_cost_function[t] = 0
        sync_cost_function[t] = 10000

    elif t.label is None:
        model_cost_function[t] = 0
        sync_cost_function[t] = 10000

    else: # Se è un task vero e proprio
        model_cost_function[t] = 10000
        sync_cost_function[t] = 0

parameters = {
    alignments.Parameters.PARAM_MODEL_COST_FUNCTION: model_cost_function,
    alignments.Parameters.PARAM_SYNC_COST_FUNCTION: sync_cost_function
}

# Eseguiamo l'allineamento
aligned_traces = alignments.apply(eventlog_traces, net.net, net.initial_marking, net.final_marking, parameters=parameters)
for i,trace in enumerate(aligned_traces):
    print(f"{i}: {trace}")

0: ['start_T1', 'end_T1', 'start_T2', 'end_T2', 'start_T5', 'end_T5', 'start_T3', 'end_T3', 'start_T4', 'end_T4', 'start_T6', 'end_T6', 'start_T6', 'end_T6', 'start_T2', 'end_T2', 'start_T5', 'end_T5', 'start_T1', 'end_T1', 'start_T6', 'end_T6', 'start_T5', 'end_T5', 'start_T4', 'end_T4', 'start_T6', 'end_T6', 'start_T3', 'end_T3', 'start_T6', 'end_T6', 'start_T6', 'end_T6', 'start_T5', 'end_T5', 'start_T4', 'end_T4', 'start_T3', 'end_T3', 'start_T6', 'end_T6', 'start_T3', 'end_T3', 'start_T3', 'end_T3', 'start_T3', 'end_T3', 'start_T3', 'end_T3', 'start_T5', 'end_T5', 'start_T6', 'end_T6', 'start_T1', 'end_T1', 'start_T2', 'end_T2', 'start_T1', 'end_T1', 'start_T1', 'end_T1', 'start_T6', 'end_T6', 'start_T6', 'end_T6', 'start_T2', 'end_T2', 'start_T5', 'end_T5', 'start_T1', 'end_T1', 'start_T4', 'end_T4', 'start_T3', 'end_T3', 'start_T4', 'end_T4', 'start_T6', 'end_T6', 'start_T6', 'end_T6', 'start_T4', 'end_T4', 'start_T6', 'end_T6', 'start_T6', 'end_T6', 'start_T1', 'end_T1', 'start

aligning log, completed variants ::   0%|          | 0/12 [00:00<?, ?it/s]

0: {'alignment': [('>>', 'start_L0'), ('>>', 'start_P1'), ('>>', 'start_L8'), ('>>', 'start_L3'), ('>>', 'start_P4'), ('start_T1', 'start_T1'), ('end_T1', 'end_T1'), ('start_T2', 'start_T2'), ('end_T2', 'end_T2'), ('start_T5', 'start_T5'), ('end_T5', 'end_T5'), ('>>', 'end_P4'), ('>>', 'end_L3'), ('>>', 'start_L5'), ('>>', 'start_P6'), ('>>', 'start_L7'), ('start_T3', 'start_T3'), ('end_T3', 'end_T3'), ('start_T4', 'start_T4'), ('end_T4', 'end_T4'), ('>>', 'end_L7'), ('>>', 'end_P6'), ('>>', 'start_L10'), ('>>', 'end_L5'), ('start_T6', 'start_T6'), ('end_T6', 'end_T6'), ('>>', 'back_L10'), ('start_T6', 'start_T6'), ('end_T6', 'end_T6'), ('>>', 'end_L10'), ('>>', 'end_L8'), ('>>', 'end_P1'), ('>>', 'back_L0'), ('>>', 'start_P1'), ('>>', 'start_L3'), ('>>', 'start_P4'), ('>>', 'start_L8'), ('start_T2', 'start_T2'), ('end_T2', 'end_T2'), ('start_T5', 'start_T5'), ('end_T5', 'end_T5'), ('start_T1', 'start_T1'), ('>>', 'start_L10'), ('end_T1', 'end_T1'), ('>>', 'end_P4'), ('>>', 'end_L3'), 

In [13]:
check = ["P", "X", "L"]
start = tuple(["start_" + c for c in check])
end = tuple(["end_" + c for c in check if c!="L"]) # si escludono gli end loop
back_loop = tuple(["back_L"])

# Puliamo l'allineamento e otteniamo la traccia allineata più vicina a quella generata
aligned_traces_cleaned = []
for a_trace in aligned_traces:
    trace_cleaned = []
    trace = a_trace['alignment']

    print(trace)

    for _, net_step in trace: #trans --> trans step , net --> real petri net step
        if net_step != None:
            if not (net_step.startswith(start) or net_step.startswith(end) or net_step.startswith(back_loop)) and not net_step=='>>':
                trace_cleaned.append(net_step)

    aligned_traces_cleaned.append(trace_cleaned)


for i,trace in enumerate(aligned_traces_cleaned):
    print(f"{i}: {trace}")

# Codifichiamo le tracce allineati (per poi poterle confrontare con quelle generate dal transformer)
aligned_traceEncoded_regions, aligned_traceEncoded_tasks = getEncoding(aligned_traces_cleaned, net.regions, net.tasks, net.open_clauses, net.end_clauses)
df_aligned_traces = pd.concat([aligned_traceEncoded_regions, aligned_traceEncoded_tasks], axis=0)
print(df_aligned_traces)


[('>>', 'start_L0'), ('>>', 'start_P1'), ('>>', 'start_L8'), ('>>', 'start_L3'), ('>>', 'start_P4'), ('start_T1', 'start_T1'), ('end_T1', 'end_T1'), ('start_T2', 'start_T2'), ('end_T2', 'end_T2'), ('start_T5', 'start_T5'), ('end_T5', 'end_T5'), ('>>', 'end_P4'), ('>>', 'end_L3'), ('>>', 'start_L5'), ('>>', 'start_P6'), ('>>', 'start_L7'), ('start_T3', 'start_T3'), ('end_T3', 'end_T3'), ('start_T4', 'start_T4'), ('end_T4', 'end_T4'), ('>>', 'end_L7'), ('>>', 'end_P6'), ('>>', 'start_L10'), ('>>', 'end_L5'), ('start_T6', 'start_T6'), ('end_T6', 'end_T6'), ('>>', 'back_L10'), ('start_T6', 'start_T6'), ('end_T6', 'end_T6'), ('>>', 'end_L10'), ('>>', 'end_L8'), ('>>', 'end_P1'), ('>>', 'back_L0'), ('>>', 'start_P1'), ('>>', 'start_L3'), ('>>', 'start_P4'), ('>>', 'start_L8'), ('start_T2', 'start_T2'), ('end_T2', 'end_T2'), ('start_T5', 'start_T5'), ('end_T5', 'end_T5'), ('start_T1', 'start_T1'), ('>>', 'start_L10'), ('end_T1', 'end_T1'), ('>>', 'end_P4'), ('>>', 'end_L3'), ('start_T6', 'sta

In [14]:
# Creo una lista delle tracce codificate (ogni traccia è una lista dove ogni elemento è una colonna del df, ossia uno step) --> più facili da confrontare quando calcoliamo la distanza
aligned_traces_encoded = []
aligned_trace_encoded = []
for element in df_aligned_traces.T.values:
    element = element.tolist()
    aligned_trace_encoded.append(element)
    if element == [0] * (num_regions+num_tasks):
        aligned_traces_encoded.append(aligned_trace_encoded)
        aligned_trace_encoded = []

# Andiamo a calcolare il costo con la edit distance (weighted_levenshtein)
for i in range(len(aligned_traces_encoded)):
    generated_t = traces_generated[i]
    aligned_t = aligned_traces_encoded[i]

    gen_tuples = [tuple(step) for step in generated_t]
    aligned_tuples = [tuple(step) for step in aligned_t]

    total_possible_column = set(gen_tuples + aligned_tuples) # Prendo tutte le possibili colonne per andare a creare il dizionario delle sostituzioni

    # Se volessi dizionario delle distanze bisognerebbe usare questo pezzo di codice (adesso usiamo distanza di hamming di base nel codice)
    '''cost_sub = {}
    for e1 in total_possible_column:
        for e2 in total_possible_column:
            if e1 != e2:
                counter = 0
                for i in range(len(e1)):
                    if e1[i] != e2[i]:
                        counter+=1
                cost_sub[(e1,e2)] = counter'''

    cost = edit_distance_weighted_levenshtein(generated_t, aligned_t, num_regions+num_tasks, num_regions+num_tasks, hamming_distance) # Utilizziamo la distanza di hamming al momento
    print(cost)

#costo = edit_distance_weighted_levenshtein(traccia_ai, traccia_pulita, cost_ins, cost_del, cost_sub)
#print(f"Costo di correzione totale: {costo}")

185.0
37.0
156.0
76.0
93.0
168.0
38.0
174.0
80.0
18.0
66.0
43.0


In [14]:
print()